# B2 — Period analysis and measurement

Continues from B1. Three threads:

1. Whether the divergence from Bryzgalova et al. (2023) is explained by
   sample period — their window is Nov 2019 to Jun 2021, roughly 15% of this
   sample.
2. A discontinuity in CBOE's professional-customer classification in 2015,
   established through four independent measures.
3. Prior earnings volatility, the second uncertainty measure named in RQ1.

**Requires A1 to have been run.** This notebook only reads intermediate
files; it never rebuilds them. If a pipeline script changes, re-run the
corresponding step in A1 first.

## Setup

In [1]:
import sys
from pathlib import Path

def find_src_dir(start: Path = None) -> Path:
    current = start or Path.cwd()
    for _ in range(5):
        candidate = current / "src"
        if candidate.exists():
            return candidate
        current = current.parent
    raise FileNotFoundError("Could not locate a 'src' folder above the current directory.")

SRC_DIR = find_src_dir()
sys.path.append(str(SRC_DIR))

import polars as pl
import statsmodels.formula.api as smf
from paths import DATA_DIR, IBES_DIR

## Era comparison — is the divergence explained by period?

Bryzgalova, Pavlova & Sikorskaya (2023) report retail crowding into options
in the two weeks preceding announcements. This sample shows no such buildup.
If the pattern appears inside their window and not outside it, the
discrepancy is period rather than method.

Volume is normalised by each era's own far-from-event baseline, so eras with
very different absolute volumes remain comparable.

In [2]:
from analysis.event_window_profile import compare_eras_profile

prof = compare_eras_profile(window=30)

piv = (
    prof.filter(pl.col("rel_day").is_between(-20, 10))
    .pivot(values="vol_vs_baseline", index="rel_day", on="era")
    .sort("rel_day")
)
with pl.Config(tbl_rows=-1, float_precision=2):
    print(piv)

Matched 72,435 of 123,531 events to CBOE trading data
Matched 14,829 of 23,903 events to CBOE trading data
Matched 10,006 of 15,576 events to CBOE trading data
shape: (31, 4)
┌─────────┬──────────┬────────────┬───────────┐
│ rel_day ┆ pre_boom ┆ bpz_window ┆ post_boom │
│ ---     ┆ ---      ┆ ---        ┆ ---       │
│ i64     ┆ f64      ┆ f64        ┆ f64       │
╞═════════╪══════════╪════════════╪═══════════╡
│ -20     ┆ 0.97     ┆ 0.96       ┆ 0.97      │
│ -19     ┆ 0.99     ┆ 0.96       ┆ 0.97      │
│ -18     ┆ 0.99     ┆ 0.96       ┆ 1.01      │
│ -17     ┆ 0.98     ┆ 0.98       ┆ 0.97      │
│ -16     ┆ 0.97     ┆ 0.96       ┆ 0.94      │
│ -15     ┆ 0.96     ┆ 0.91       ┆ 0.89      │
│ -14     ┆ 0.98     ┆ 0.94       ┆ 0.95      │
│ -13     ┆ 0.99     ┆ 1.04       ┆ 0.95      │
│ -12     ┆ 0.98     ┆ 0.98       ┆ 0.95      │
│ -11     ┆ 0.96     ┆ 0.95       ┆ 0.88      │
│ -10     ┆ 0.98     ┆ 0.94       ┆ 0.87      │
│ -9      ┆ 1.03     ┆ 0.96       ┆ 0.98      │
│ -8     

In [3]:
from analysis.event_window_profile import compare_eras_did

did_by_era = compare_eras_did()
with pl.Config(tbl_rows=-1, float_precision=4):
    print(did_by_era)

shape: (15, 5)
┌────────────┬─────────┬─────────┬─────────┬────────┐
│ era        ┆ outcome ┆ coef    ┆ p_value ┆ n_obs  │
│ ---        ┆ ---     ┆ ---     ┆ ---     ┆ ---    │
│ str        ┆ str     ┆ f64     ┆ f64     ┆ i64    │
╞════════════╪═════════╪═════════╪═════════╪════════╡
│ pre_boom   ┆ otm     ┆ -0.0014 ┆ 0.5615  ┆ 238600 │
│ pre_boom   ┆ otm_put ┆ 0.0098  ┆ 0.0000  ┆ 238600 │
│ pre_boom   ┆ lt_100  ┆ 0.0474  ┆ 0.0000  ┆ 238600 │
│ pre_boom   ┆ call    ┆ -0.0082 ┆ 0.0011  ┆ 238600 │
│ pre_boom   ┆ open    ┆ 0.0368  ┆ 0.0000  ┆ 238600 │
│ bpz_window ┆ otm     ┆ 0.0542  ┆ 0.0000  ┆ 46530  │
│ bpz_window ┆ otm_put ┆ 0.0174  ┆ 0.0004  ┆ 46530  │
│ bpz_window ┆ lt_100  ┆ 0.0180  ┆ 0.0000  ┆ 46530  │
│ bpz_window ┆ call    ┆ 0.0023  ┆ 0.6730  ┆ 46530  │
│ bpz_window ┆ open    ┆ 0.0270  ┆ 0.0000  ┆ 46530  │
│ post_boom  ┆ otm     ┆ 0.0394  ┆ 0.0000  ┆ 32060  │
│ post_boom  ┆ otm_put ┆ -0.0020 ┆ 0.7327  ┆ 32060  │
│ post_boom  ┆ lt_100  ┆ 0.0404  ┆ 0.0000  ┆ 32060  │
│ post_boom  

## Year by year — sharp break or gradual drift?

Three coarse era bins can show that something changed but not whether it was
a break or a trend, and that distinction matters for interpretation.

In [4]:
from analysis.event_window_profile import compare_eras_did, summarise_eras_profile, yearly_eras

did_yearly = compare_eras_did(outcomes=["otm", "otm_put", "lt_100", "call", "open"], eras=yearly_eras())

pivot = did_yearly.pivot(values="coef", index="era", on="outcome").sort("era")
with pl.Config(tbl_rows=-1, float_precision=4):
    print(pivot)

shape: (12, 6)
┌──────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│ era  ┆ otm     ┆ otm_put ┆ lt_100  ┆ call    ┆ open    │
│ ---  ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---     │
│ str  ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ f64     │
╞══════╪═════════╪═════════╪═════════╪═════════╪═════════╡
│ 2011 ┆ -0.0267 ┆ -0.0185 ┆ 0.0420  ┆ 0.0207  ┆ 0.0381  │
│ 2012 ┆ -0.0243 ┆ -0.0043 ┆ 0.0452  ┆ 0.0092  ┆ 0.0421  │
│ 2013 ┆ -0.0308 ┆ -0.0056 ┆ 0.0581  ┆ -0.0037 ┆ 0.0628  │
│ 2014 ┆ -0.0355 ┆ -0.0003 ┆ 0.0729  ┆ -0.0033 ┆ 0.0420  │
│ 2015 ┆ -0.0370 ┆ -0.0030 ┆ 0.0898  ┆ -0.0159 ┆ 0.0216  │
│ 2016 ┆ 0.0194  ┆ 0.0161  ┆ 0.0683  ┆ -0.0079 ┆ 0.0269  │
│ 2017 ┆ 0.0259  ┆ 0.0397  ┆ 0.0196  ┆ -0.0451 ┆ -0.0038 │
│ 2018 ┆ 0.0471  ┆ 0.0368  ┆ 0.0018  ┆ -0.0205 ┆ 0.0150  │
│ 2019 ┆ 0.0497  ┆ 0.0289  ┆ -0.0107 ┆ -0.0102 ┆ -0.0007 │
│ 2020 ┆ 0.0620  ┆ 0.0237  ┆ 0.0245  ┆ 0.0008  ┆ 0.0526  │
│ 2021 ┆ 0.0424  ┆ 0.0060  ┆ 0.0285  ┆ 0.0054  ┆ 0.0001  │
│ 2022 ┆ 0.0362  ┆ -0.0106 ┆ 0.0473  ┆ 0.

`pre_window_mean` covers days −10 to −3, where a two-week buildup would
appear if there were one.

In [5]:
prof_yearly = summarise_eras_profile(window=30, eras=yearly_eras())
with pl.Config(tbl_rows=-1, float_precision=3):
    print(prof_yearly)

Matched 7,899 of 13,156 events to CBOE trading data
Matched 7,604 of 13,044 events to CBOE trading data
Matched 7,848 of 13,190 events to CBOE trading data
Matched 8,448 of 14,154 events to CBOE trading data
Matched 8,143 of 14,705 events to CBOE trading data
Matched 8,006 of 14,597 events to CBOE trading data
Matched 8,264 of 14,213 events to CBOE trading data
Matched 8,569 of 14,183 events to CBOE trading data
Matched 8,845 of 14,328 events to CBOE trading data
Matched 8,835 of 14,364 events to CBOE trading data
Matched 9,928 of 15,403 events to CBOE trading data
Matched 4,881 of 7,673 events to CBOE trading data
shape: (12, 7)
┌──────┬─────────────────┬────────────┬────────────┬────────────┬──────────────────┬───────────────┐
│ era  ┆ pre_window_mean ┆ day_minus1 ┆ day0_ratio ┆ day1_ratio ┆ post_window_mean ┆ n_events_day0 │
│ ---  ┆ ---             ┆ ---        ┆ ---        ┆ ---        ┆ ---              ┆ ---           │
│ str  ┆ f64             ┆ f64        ┆ f64        ┆ f64   

In [6]:
from analysis.build_results_figures import fig4_coefficients_by_year
fig4_coefficients_by_year(outcomes=["otm", "otm_put", "lt_100", "open"])

  wrote fig4_coefficients_by_year.png


## What changed in 2016 — retail or professional customers?

A difference-in-differences coefficient cannot say which group moved. This
decomposes the year-by-year series into its underlying levels.

Volume is included because a compositional explanation (who is classified as
a customer) and a behavioural one (how customers trade) have different
signatures.

In [7]:
from analysis.event_window_profile import decompose_break_by_year

dec = decompose_break_by_year(outcome="otm", eras=yearly_eras())

print("=== Event shift by group (near-event share minus baseline share) ===")
with pl.Config(tbl_rows=-1, float_precision=4):
    print(dec.pivot(values="event_shift", index="era", on="participant_group").sort("era"))

print("\n=== Baseline OTM share by group ===")
with pl.Config(tbl_rows=-1, float_precision=4):
    print(dec.pivot(values="baseline_share", index="era", on="participant_group").sort("era"))

print("\n=== Mean daily volume by group (baseline) ===")
with pl.Config(tbl_rows=-1, float_precision=1):
    print(dec.pivot(values="baseline_daily_vol", index="era", on="participant_group").sort("era"))

=== Event shift by group (near-event share minus baseline share) ===
shape: (12, 3)
┌──────┬─────────┬─────────┐
│ era  ┆ procust ┆ retail  │
│ ---  ┆ ---     ┆ ---     │
│ str  ┆ f64     ┆ f64     │
╞══════╪═════════╪═════════╡
│ 2011 ┆ 0.0238  ┆ -0.0029 │
│ 2012 ┆ 0.0266  ┆ 0.0023  │
│ 2013 ┆ 0.0354  ┆ 0.0046  │
│ 2014 ┆ 0.0541  ┆ 0.0186  │
│ 2015 ┆ 0.0499  ┆ 0.0128  │
│ 2016 ┆ -0.0028 ┆ 0.0166  │
│ 2017 ┆ -0.0086 ┆ 0.0173  │
│ 2018 ┆ -0.0196 ┆ 0.0275  │
│ 2019 ┆ -0.0405 ┆ 0.0092  │
│ 2020 ┆ -0.0447 ┆ 0.0173  │
│ 2021 ┆ -0.0313 ┆ 0.0111  │
│ 2022 ┆ -0.0244 ┆ 0.0119  │
└──────┴─────────┴─────────┘

=== Baseline OTM share by group ===
shape: (12, 3)
┌──────┬─────────┬────────┐
│ era  ┆ procust ┆ retail │
│ ---  ┆ ---     ┆ ---    │
│ str  ┆ f64     ┆ f64    │
╞══════╪═════════╪════════╡
│ 2011 ┆ 0.4674  ┆ 0.5776 │
│ 2012 ┆ 0.3956  ┆ 0.5769 │
│ 2013 ┆ 0.4348  ┆ 0.5638 │
│ 2014 ┆ 0.4754  ┆ 0.5561 │
│ 2015 ┆ 0.5089  ┆ 0.5733 │
│ 2016 ┆ 0.5685  ┆ 0.5902 │
│ 2017 ┆ 0.5868  ┆ 0.5737 │
│ 2018

### Coverage across the whole dataset

No event matching and no sample conditioning, so this is a cleaner measure
than the event-matched panel. `procust_coverage` is the fraction of
ticker-days with any professional-customer activity.

In [8]:
daily = pl.concat([pl.read_parquet(f) for f in sorted((DATA_DIR / "cboe_daily_retail").glob("*.parquet"))])

by_year = (
    daily.with_columns(pl.col("quote_date").dt.year().alias("year"))
    .group_by("year")
    .agg(
        pl.col("retail_vol_total").sum().alias("retail_vol"),
        pl.col("procust_vol_total").sum().alias("procust_vol"),
        pl.col("procust_vol_total").filter(pl.col("procust_vol_total") > 0).len().alias("procust_active_days"),
        pl.len().alias("ticker_days"),
    )
    .with_columns(
        (pl.col("procust_vol") / (pl.col("procust_vol") + pl.col("retail_vol"))).alias("procust_share"),
        (pl.col("procust_active_days") / pl.col("ticker_days")).alias("procust_coverage"),
    )
    .sort("year")
)
with pl.Config(tbl_rows=-1, float_precision=4):
    print(by_year)

shape: (12, 7)
┌──────┬────────────┬─────────────┬─────────────────┬─────────────┬───────────────┬────────────────┐
│ year ┆ retail_vol ┆ procust_vol ┆ procust_active_ ┆ ticker_days ┆ procust_share ┆ procust_covera │
│ ---  ┆ ---        ┆ ---         ┆ days            ┆ ---         ┆ ---           ┆ ge             │
│ i32  ┆ i64        ┆ i64         ┆ ---             ┆ u32         ┆ f64           ┆ ---            │
│      ┆            ┆             ┆ u32             ┆             ┆               ┆ f64            │
╞══════╪════════════╪═════════════╪═════════════════╪═════════════╪═══════════════╪════════════════╡
│ 2011 ┆ 830598782  ┆ 32082139    ┆ 157853          ┆ 537987      ┆ 0.0372        ┆ 0.2934         │
│ 2012 ┆ 802136023  ┆ 36607272    ┆ 151429          ┆ 503887      ┆ 0.0436        ┆ 0.3005         │
│ 2013 ┆ 805611470  ┆ 34161056    ┆ 152202          ┆ 520012      ┆ 0.0407        ┆ 0.2927         │
│ 2014 ┆ 932267883  ┆ 29637854    ┆ 137375          ┆ 563196      ┆ 0.0308  

### Is the drop uniform or concentrated?

A reporting threshold would spare large tickers; a definitional change would
not. Splitting by retail-volume quartile within each year distinguishes
them.

In [9]:
tick_year = (
    daily.with_columns(pl.col("quote_date").dt.year().alias("year"))
    .group_by(["underlying_symbol", "year"])
    .agg(
        pl.col("retail_vol_total").sum().alias("retail_vol"),
        (pl.col("procust_vol_total") > 0).sum().alias("procust_days"),
        pl.len().alias("days"),
    )
)

q = (
    tick_year.with_columns(
        pl.col("retail_vol").qcut(4, labels=["Q1_small","Q2","Q3","Q4_large"],
                                  allow_duplicates=True).over("year").alias("vol_q")
    )
    .group_by(["year", "vol_q"])
    .agg((pl.col("procust_days").sum() / pl.col("days").sum()).alias("procust_coverage"))
)

with pl.Config(tbl_rows=-1, float_precision=3):
    print(q.pivot(values="procust_coverage", index="year", on="vol_q").sort("year"))

shape: (12, 5)
┌──────┬──────────┬───────┬───────┬──────────┐
│ year ┆ Q1_small ┆ Q3    ┆ Q2    ┆ Q4_large │
│ ---  ┆ ---      ┆ ---   ┆ ---   ┆ ---      │
│ i32  ┆ f64      ┆ f64   ┆ f64   ┆ f64      │
╞══════╪══════════╪═══════╪═══════╪══════════╡
│ 2011 ┆ 0.050    ┆ 0.186 ┆ 0.084 ┆ 0.565    │
│ 2012 ┆ 0.064    ┆ 0.177 ┆ 0.101 ┆ 0.536    │
│ 2013 ┆ 0.078    ┆ 0.174 ┆ 0.092 ┆ 0.502    │
│ 2014 ┆ 0.046    ┆ 0.138 ┆ 0.067 ┆ 0.445    │
│ 2015 ┆ 0.014    ┆ 0.049 ┆ 0.021 ┆ 0.274    │
│ 2016 ┆ 0.012    ┆ 0.042 ┆ 0.016 ┆ 0.286    │
│ 2017 ┆ 0.012    ┆ 0.052 ┆ 0.016 ┆ 0.285    │
│ 2018 ┆ 0.009    ┆ 0.126 ┆ 0.032 ┆ 0.379    │
│ 2019 ┆ 0.007    ┆ 0.074 ┆ 0.020 ┆ 0.304    │
│ 2020 ┆ 0.012    ┆ 0.079 ┆ 0.026 ┆ 0.433    │
│ 2021 ┆ 0.017    ┆ 0.116 ┆ 0.045 ┆ 0.451    │
│ 2022 ┆ 0.022    ┆ 0.092 ┆ 0.035 ┆ 0.438    │
└──────┴──────────┴───────┴───────┴──────────┘


### Where did the lost volume go?

If participants were reclassified rather than losing the designation, the
lost volume should reappear in another category. Market-maker share is the
control: market makers are defined by exchange role rather than order counts,
so their share should be stable if this is a Professional-definition
change.

In [10]:
by_year_all = (
    daily.with_columns(pl.col("quote_date").dt.year().alias("year"))
    .group_by("year")
    .agg(
        pl.col("retail_vol_total").sum().alias("retail"),
        pl.col("procust_vol_total").sum().alias("procust"),
        pl.col("firm_vol_total").sum().alias("firm"),
        pl.col("bd_vol_total").sum().alias("bd"),
        pl.col("mm_vol_total").sum().alias("mm"),
    )
    .sort("year")
)

print("=== Year-on-year change ===")
with pl.Config(tbl_rows=-1):
    print(
        by_year_all.with_columns([
            (pl.col(c) - pl.col(c).shift(1)).alias(f"d_{c}")
            for c in ["retail", "procust", "firm", "bd", "mm"]
        ]).select(["year", "d_retail", "d_procust", "d_firm", "d_bd", "d_mm"])
    )

print("\n=== Shares ===")
total = pl.sum_horizontal(["retail", "procust", "firm", "bd", "mm"])
shares = by_year_all.with_columns([
    (pl.col(c) / total).alias(f"share_{c}") for c in ["retail", "procust", "firm", "bd", "mm"]
]).select(["year"] + [f"share_{c}" for c in ["retail", "procust", "firm", "bd", "mm"]])

with pl.Config(tbl_rows=-1, float_precision=4):
    print(shares)

=== Year-on-year change ===
shape: (12, 6)
┌──────┬────────────┬───────────┬────────────┬──────────┬────────────┐
│ year ┆ d_retail   ┆ d_procust ┆ d_firm     ┆ d_bd     ┆ d_mm       │
│ ---  ┆ ---        ┆ ---       ┆ ---        ┆ ---      ┆ ---        │
│ i32  ┆ i64        ┆ i64       ┆ i64        ┆ i64      ┆ i64        │
╞══════╪════════════╪═══════════╪════════════╪══════════╪════════════╡
│ 2011 ┆ null       ┆ null      ┆ null       ┆ null     ┆ null       │
│ 2012 ┆ -28462759  ┆ 4525133   ┆ -81699615  ┆ -7712861 ┆ -68491442  │
│ 2013 ┆ 3475447    ┆ -2446216  ┆ 15350965   ┆ -5793884 ┆ 10701048   │
│ 2014 ┆ 126656413  ┆ -4523202  ┆ 44214611   ┆ -5178775 ┆ 84016881   │
│ 2015 ┆ -108383191 ┆ -17209189 ┆ -26652173  ┆ -779715  ┆ -150798822 │
│ 2016 ┆ 7987111    ┆ 6233907   ┆ 25094612   ┆ 1786638  ┆ -57446214  │
│ 2017 ┆ 93650906   ┆ 16611127  ┆ 33216706   ┆ -6016113 ┆ 58607046   │
│ 2018 ┆ 93209634   ┆ 5181190   ┆ 21478562   ┆ 10791078 ┆ 170291800  │
│ 2019 ┆ -76909093  ┆ -8221019  ┆ 

### Did participants restructure orders?

Professional status depends on placing more than 390 orders per day on
average, and public customers receive priority and fee benefits, so there is
an incentive to stay below the threshold. Someone doing so would place fewer,
larger orders.

CBOE reports volume and transaction counts, not orders, so the threshold
cannot be observed directly. Contracts per transaction is the closest
available proxy. Market makers again serve as the control.

In [11]:
from analysis.analyse_order_size import order_size_by_year, order_size_change

with pl.Config(tbl_rows=-1, float_precision=2):
    print(order_size_by_year())
    print()
    print(order_size_change())

shape: (12, 6)
┌──────┬─────────────┬──────────────┬───────────┬─────────┬─────────┐
│ year ┆ retail_size ┆ procust_size ┆ firm_size ┆ bd_size ┆ mm_size │
│ ---  ┆ ---         ┆ ---          ┆ ---       ┆ ---     ┆ ---     │
│ i32  ┆ f64         ┆ f64          ┆ f64       ┆ f64     ┆ f64     │
╞══════╪═════════════╪══════════════╪═══════════╪═════════╪═════════╡
│ 2011 ┆ 18.00       ┆ 10.83        ┆ 45.41     ┆ 70.51   ┆ 19.55   │
│ 2012 ┆ 14.67       ┆ 10.79        ┆ 44.42     ┆ 105.29  ┆ 16.83   │
│ 2013 ┆ 16.18       ┆ 13.08        ┆ 67.35     ┆ 117.44  ┆ 16.69   │
│ 2014 ┆ 15.32       ┆ 13.16        ┆ 82.28     ┆ 114.48  ┆ 14.77   │
│ 2015 ┆ 16.66       ┆ 14.20        ┆ 85.33     ┆ 104.77  ┆ 14.89   │
│ 2016 ┆ 16.23       ┆ 18.60        ┆ 90.29     ┆ 96.62   ┆ 14.27   │
│ 2017 ┆ 13.78       ┆ 31.79        ┆ 99.92     ┆ 51.56   ┆ 12.72   │
│ 2018 ┆ 12.39       ┆ 30.15        ┆ 87.30     ┆ 62.16   ┆ 11.73   │
│ 2019 ┆ 10.45       ┆ 20.35        ┆ 87.02     ┆ 63.64   ┆ 9.68    │
│ 202

### Exchange classification vs. the small-trade proxy

`recall` is the share of true retail volume a small-trade filter would
capture; `contamination_lb` is a lower bound on how much small customer
volume is professional; `classifiable_share` is the fraction of participant
volume that can be size-classified at all.

Only the two customer categories carry size tiers, so a full small-trade
proxy across all participant types cannot be reconstructed from these data.

In [12]:
from analysis.compare_retail_proxies import compare_proxies

with pl.Config(tbl_rows=-1, tbl_cols=10, float_precision=4):
    print(compare_proxies())

shape: (12, 6)
┌──────┬────────┬──────────────────┬────────────────────┬────────────┬─────────────┐
│ year ┆ recall ┆ contamination_lb ┆ classifiable_share ┆ retail_vol ┆ procust_vol │
│ ---  ┆ ---    ┆ ---              ┆ ---                ┆ ---        ┆ ---         │
│ i32  ┆ f64    ┆ f64              ┆ f64                ┆ i64        ┆ i64         │
╞══════╪════════╪══════════════════╪════════════════════╪════════════╪═════════════╡
│ 2011 ┆ 0.5018 ┆ 0.0577           ┆ 0.3758             ┆ 830598782  ┆ 32082139    │
│ 2012 ┆ 0.5334 ┆ 0.0572           ┆ 0.3968             ┆ 802136023  ┆ 36607272    │
│ 2013 ┆ 0.4959 ┆ 0.0527           ┆ 0.3933             ┆ 805611470  ┆ 34161056    │
│ 2014 ┆ 0.4940 ┆ 0.0406           ┆ 0.4041             ┆ 932267883  ┆ 29637854    │
│ 2015 ┆ 0.4759 ┆ 0.0193           ┆ 0.4027             ┆ 823884692  ┆ 12428665    │
│ 2016 ┆ 0.4589 ┆ 0.0234           ┆ 0.4128             ┆ 831871803  ┆ 18662572    │
│ 2017 ┆ 0.4838 ┆ 0.0230           ┆ 0.4258       

## Headline results — pooled vs. 2016-onward primary sample

Because the professional-customer category is not consistently defined before
2015, full-sample estimates pool two differently-composed control groups.
This quantifies how much that matters for each outcome.

In [13]:
from analysis.build_results_tables import table6_period_comparison

cmp = table6_period_comparison()
with pl.Config(tbl_rows=-1, tbl_cols=10, float_precision=4):
    print(cmp)

shape: (7, 8)
┌──────────┬─────────────┬──────────┬──────────┬────────────┬────────────┬────────────┬────────────┐
│ outcome  ┆ pooled_coef ┆ pooled_p ┆ pooled_n ┆ from2016_c ┆ from2016_p ┆ from2016_n ┆ difference │
│ ---      ┆ ---         ┆ ---      ┆ ---      ┆ oef        ┆ ---        ┆ ---        ┆ ---        │
│ str      ┆ f64         ┆ f64      ┆ i64      ┆ ---        ┆ f64        ┆ i64        ┆ f64        │
│          ┆             ┆          ┆          ┆ f64        ┆            ┆            ┆            │
╞══════════╪═════════════╪══════════╪══════════╪════════════╪════════════╪════════════╪════════════╡
│ otm      ┆ 0.0209      ┆ 0.0000   ┆ 185416   ┆ 0.0388     ┆ 0.0000     ┆ 91124      ┆ 0.0180     │
│ otm_put  ┆ 0.0182      ┆ 0.0000   ┆ 185416   ┆ 0.0226     ┆ 0.0000     ┆ 91124      ┆ 0.0044     │
│ otm_call ┆ 0.0027      ┆ 0.1597   ┆ 185416   ┆ 0.0162     ┆ 0.0000     ┆ 91124      ┆ 0.0136     │
│ itm      ┆ -0.0123     ┆ 0.0000   ┆ 185416   ┆ -0.0170    ┆ 0.0000     ┆ 91

In [14]:
from analysis.build_results_tables import table3_dispersion, PRIMARY_SAMPLE_START

print("=== Dispersion, 2016 onward ===")
disp_restricted = table3_dispersion(date_from=PRIMARY_SAMPLE_START)
with pl.Config(tbl_rows=-1, tbl_cols=12, float_precision=4):
    print(disp_restricted)

=== Dispersion, 2016 onward ===
shape: (7, 9)
┌──────────┬──────────┬────────┬──────────┬────────┬────────────┬─────────┬──────────┬────────┐
│ outcome  ┆ did_base ┆ p_base ┆ did_ctrl ┆ p_ctrl ┆ cross_ctrl ┆ p_cross ┆ size_did ┆ p_size │
│ ---      ┆ ---      ┆ ---    ┆ ---      ┆ ---    ┆ ---        ┆ ---     ┆ ---      ┆ ---    │
│ str      ┆ f64      ┆ f64    ┆ f64      ┆ f64    ┆ f64        ┆ f64     ┆ f64      ┆ f64    │
╞══════════╪══════════╪════════╪══════════╪════════╪════════════╪═════════╪══════════╪════════╡
│ otm      ┆ -0.0079  ┆ 0.0086 ┆ -0.0038  ┆ 0.2099 ┆ -0.0038    ┆ 0.0618  ┆ 0.0195   ┆ 0.0000 │
│ otm_put  ┆ -0.0074  ┆ 0.0086 ┆ -0.0045  ┆ 0.1202 ┆ 0.0031     ┆ 0.1274  ┆ 0.0152   ┆ 0.0000 │
│ otm_call ┆ -0.0004  ┆ 0.8915 ┆ 0.0007   ┆ 0.8277 ┆ -0.0069    ┆ 0.0013  ┆ 0.0043   ┆ 0.1842 │
│ itm      ┆ 0.0010   ┆ 0.7026 ┆ -0.0019  ┆ 0.4526 ┆ 0.0020     ┆ 0.2303  ┆ -0.0153  ┆ 0.0000 │
│ lt_100   ┆ 0.0026   ┆ 0.2501 ┆ 0.0000   ┆ 0.9848 ┆ -0.0039    ┆ 0.0533  ┆ -0.0125  ┆ 0.0

In [15]:
from analysis.event_window_profile import build_diff_in_diff_panel, add_market_cap, run_dispersion_regression

panel = add_market_cap(build_diff_in_diff_panel(outcome="otm", date_from=PRIMARY_SAMPLE_START))
m = run_dispersion_regression(panel, spec="triple", outcome_var="log_volume",
                              cluster_by="ticker", controls=["log_mktcap"])
print(m.summary().tables[1])

Matched 57,328 of 94,761 events to CBOE trading data
Panel has 177,247 rows (up to 2 periods x 2 groups per event)
Market cap matched for 176,608 of 177,247 panel rows (99.6%)
  Dropped 639 rows missing control values (176,608 remain)
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.6141      0.027     22.935      0.000       0.562       0.667
dispersion                0.1492      0.017      8.702      0.000       0.116       0.183
treat                     4.0571      0.017    236.488      0.000       4.023       4.091
dispersion:treat          0.0387      0.013      2.928      0.003       0.013       0.065
post                      1.0471      0.019     53.896      0.000       1.009       1.085
dispersion:post          -0.0112      0.017     -0.673      0.501      -0.044       0.021
treat:post               -0.7907      0.021  

## RQ1's second uncertainty measure — prior earnings volatility

Where dispersion captures how much analysts disagree ahead of an
announcement, this captures how unpredictable a firm's earnings have
historically been. Computed as the rolling standard deviation of past scaled
earnings surprises, strictly backward-looking.

The proposal treats the two as interchangeable proxies for earnings-related
uncertainty. They are not.

In [16]:
for var in ["dispersion_scaled", "earnings_volatility"]:
    m = run_dispersion_regression(panel, spec="triple", cluster_by="ticker",
                                  controls=["log_mktcap"], uncertainty_var=var)
    print(f"=== {var} ===")
    print(f"  triple      = {m.params['dispersion:treat:post']:+.4f} (p={m.pvalues['dispersion:treat:post']:.4f})")
    print(f"  cross-sect. = {m.params['dispersion:treat']:+.4f} (p={m.pvalues['dispersion:treat']:.4f})")

  Dropped 639 rows missing control values (176,608 remain)
=== dispersion_scaled ===
  triple      = -0.0038 (p=0.2099)
  cross-sect. = -0.0038 (p=0.0618)
  Dropped 586 rows missing control values (168,676 remain)
=== earnings_volatility ===
  triple      = -0.0134 (p=0.0000)
  cross-sect. = +0.0013 (p=0.5889)


In [17]:
for var in ["dispersion_scaled", "earnings_volatility"]:
    m = run_dispersion_regression(panel, spec="triple", outcome_var="log_volume",
                                  cluster_by="ticker", controls=["log_mktcap"], uncertainty_var=var)
    print(f"{var:20s} cross-sect. = {m.params['dispersion:treat']:+.4f} (p={m.pvalues['dispersion:treat']:.4f})")

  Dropped 639 rows missing control values (176,608 remain)
dispersion_scaled    cross-sect. = +0.0387 (p=0.0034)
  Dropped 586 rows missing control values (168,676 remain)
earnings_volatility  cross-sect. = +0.0000 (p=0.9980)


### Measure diagnostics

The first version of this measure produced a coefficient of −5.34 on a
bounded outcome, traced to firms with scale-corrupted MEANEST values that
pass the ceiling filter but break the rolling window by spanning two
incompatible scales. Winsorising the surprise at the 1st/99th percentile
before computing the standard deviation fixed it.

These cells document that the measure is now well-behaved.

In [18]:
ev = pl.read_parquet(IBES_DIR / "dispersion_events.parquet")

print("=== earnings_volatility ===")
print(ev["earnings_volatility"].describe())
print("\n99th / 99.9th / max:",
      ev["earnings_volatility"].quantile(0.99),
      ev["earnings_volatility"].quantile(0.999),
      ev["earnings_volatility"].max())

print("\n=== underlying surprise_scaled ===")
print(ev["surprise_scaled"].describe())

print("\n=== largest values (should be real high-volatility firms, not corruption) ===")
print(
    ev.filter(pl.col("earnings_volatility").is_not_null())
      .sort("earnings_volatility", descending=True, nulls_last=True)
      .select(["OFTIC", "ANNDATS_ACT", "MEANEST", "ACTUAL", "surprise_scaled", "earnings_volatility"])
      .head(15)
)

=== earnings_volatility ===
shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ value    │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 137886.0 │
│ null_count ┆ 25133.0  │
│ mean       ┆ 0.556975 │
│ std        ┆ 0.710489 │
│ min        ┆ 0.0      │
│ 25%        ┆ 0.106904 │
│ 50%        ┆ 0.256466 │
│ 75%        ┆ 0.691678 │
│ max        ┆ 5.559034 │
└────────────┴──────────┘

99th / 99.9th / max: 3.1467890693409166 4.3115222937839395 5.559033831778427

=== underlying surprise_scaled ===
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 162356.0  │
│ null_count ┆ 663.0     │
│ mean       ┆ 0.006607  │
│ std        ┆ 0.996718  │
│ min        ┆ -6.2      │
│ 25%        ┆ -0.090909 │
│ 50%        ┆ 0.038168  │
│ 75%        ┆ 0.2       │
│ max        ┆ 4.2       │
└────────────┴───────────┘

=== largest values (should be real

### Robustness

Clustering, functional form, and whether the effect appears across outcomes
or only one.

In [19]:
for cb in ["event", "ticker"]:
    m = run_dispersion_regression(panel, spec="triple", cluster_by=cb,
                                  controls=["log_mktcap"], uncertainty_var="earnings_volatility")
    print(f"{cb:7s} triple = {m.params['dispersion:treat:post']:+.4f} (p={m.pvalues['dispersion:treat:post']:.4f})")

  Dropped 586 rows missing control values (168,676 remain)
event   triple = -0.0134 (p=0.0000)
  Dropped 586 rows missing control values (168,676 remain)
ticker  triple = -0.0134 (p=0.0000)


In [20]:
df = (
    panel.filter(pl.col("is_near_event") & pl.col("earnings_volatility").is_not_null())
    .with_columns(
        pl.col("earnings_volatility").qcut(4, labels=["Q1","Q2","Q3","Q4"],
                                           allow_duplicates=True).alias("ev_q")
    )
    .to_pandas()
    .dropna(subset=["log_mktcap"])
)
df["treat"] = (df["participant_group"] == "retail").astype(int)
df["log_mktcap_z"] = (df["log_mktcap"] - df["log_mktcap"].mean()) / df["log_mktcap"].std()

m = smf.ols("share ~ C(ev_q) * treat + log_mktcap_z * treat", data=df).fit(
    cov_type="cluster", cov_kwds={"groups": df["resolved_ticker"]}
)
print(m.summary().tables[1])

                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               0.5968      0.006    101.474      0.000       0.585       0.608
C(ev_q)[T.Q3]           0.0251      0.008      3.170      0.002       0.010       0.041
C(ev_q)[T.Q4]           0.0315      0.009      3.699      0.000       0.015       0.048
C(ev_q)[T.Q2]           0.0112      0.007      1.515      0.130      -0.003       0.026
treat                   0.0211      0.006      3.502      0.000       0.009       0.033
C(ev_q)[T.Q3]:treat    -0.0109      0.008     -1.351      0.177      -0.027       0.005
C(ev_q)[T.Q4]:treat    -0.0200      0.009     -2.293      0.022      -0.037      -0.003
C(ev_q)[T.Q2]:treat    -0.0065      0.007     -0.875      0.382      -0.021       0.008
log_mktcap_z           -0.0395      0.003    -11.401      0.000      -0.046      -0.033
log_mktcap_z:treat      0.0263  

In [21]:
for oc in ["otm", "otm_put", "otm_call", "lt_100", "call", "open"]:
    p = add_market_cap(build_diff_in_diff_panel(outcome=oc, date_from=PRIMARY_SAMPLE_START, verbose=False), verbose=False)
    m = run_dispersion_regression(p, spec="triple", cluster_by="ticker",
                                  controls=["log_mktcap"], uncertainty_var="earnings_volatility",
                                  verbose=False)
    print(f"{oc:9s} {m.params['dispersion:treat:post']:+.4f} (p={m.pvalues['dispersion:treat:post']:.4f})")

otm       -0.0134 (p=0.0000)
otm_put   -0.0098 (p=0.0022)
otm_call  -0.0036 (p=0.3031)
lt_100    -0.0010 (p=0.6855)
call      +0.0042 (p=0.2420)
open      -0.0049 (p=0.1033)


### Like-for-like comparison

Earnings volatility requires four prior announcements, so it is estimated on
a slightly smaller sample than dispersion. Running dispersion on the
volatility sample confirms the difference is the measure, not the sample.

In [22]:
sub = panel.filter(pl.col("earnings_volatility").is_not_null())
m = run_dispersion_regression(sub, spec="triple", cluster_by="ticker",
                              controls=["log_mktcap"], uncertainty_var="dispersion_scaled")
print(f"dispersion on the volatility sample: {m.params['dispersion:treat:post']:+.4f} "
      f"(p={m.pvalues['dispersion:treat:post']:.4f})")

  Dropped 586 rows missing control values (168,676 remain)
dispersion on the volatility sample: -0.0038 (p=0.2199)
